# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [83]:
# Write your code below.
%load_ext dotenv
%dotenv 


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [84]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [85]:
import os
from glob import glob

# Write your code below.
parquet_files=glob(os.path.join(os.getenv('PRICE_DATA'), "**/*.parquet"), recursive=True)


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [86]:
# Write your code below.

ddf = dd.read_parquet(parquet_files)

ddf


,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year
npartitions=2924,,,,,,,,,,
,datetime64[ns],float64,float64,float64,float64,float64,float64,string,string,int32
,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...


In [90]:
def add_features(df):
    df = df.sort_values ("Date")
    df["Close_lag_1"] = df["Close"].shift(1)
    df["Adj_Close_lag_1"] = df["Adj Close"].shift(1)
    df["returns"] = (df["Close"] / df["Close_lag_1"]) - 1
    df["hi_lo_range"] = df["High"] - df["Low"]
    return df

dd_feat = ddf.groupby("ticker").apply(add_features)

/var/folders/df/22hm8tb927g5tvf2nq8vq8r80000gn/T/ipykernel_59263/451368260.py:9: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = ddf.groupby("ticker").apply(add_features)


In [91]:
dd_feat

,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
npartitions=2924,,,,,,,,,,,,,,
,datetime64[ns],float64,float64,float64,float64,float64,float64,string,string,int32,float64,float64,float64,float64
,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [92]:
# Write your code below.
pd_feat = dd_feat.compute()

In [94]:
pd_feat

Date       Open       High        Low      Close  \
ticker                                                                 
MICT   34865  2012-10-17   2.100000   2.100000   2.100000   2.100000   
       34866  2012-10-18   2.100000   2.100000   2.100000   2.100000   
       34867  2012-10-19   2.100000   2.100000   2.100000   2.100000   
       34868  2012-10-22   2.100000   2.100000   2.100000   2.100000   
       34869  2012-10-23   2.100000   2.100000   2.100000   2.100000   
...                  ...        ...        ...        ...        ...   
SO     177658 2020-03-26  50.000000  55.799999  49.900002  55.139999   
       177659 2020-03-27  53.220001  58.259998  53.000000  56.009998   
       177660 2020-03-30  56.720001  58.369999  56.259998  57.990002   
       177661 2020-03-31  57.000000  57.290001  53.590000  54.139999   
       177662 2020-04-01  51.900002  52.430000  49.259998  50.139999   

               Adj Close      Volume    source ticker  Year  Close_lag_1  \
ticker                                                                     
MICT   34865    2.100000         0.0  MICT.csv   MICT  2012          NaN   
       34866    2.100000         0.0  MICT.csv   MICT  2012     2.100000   
       34867    2.100000         0.0  MICT.csv   MICT  2012     2.100000   
       34868    2.100000         0.0  MICT.csv   MICT  2012     2.100000   
       34869    2.100000         0.0  MICT.csv   MICT  2012     2.100000   
...                  ...         ...       ...    ...   ...          ...   
SO     177658  55.139999   8647300.0    SO.csv     SO  2020    50.139999   
       177659  56.009998  10065000.0    SO.csv     SO  2020    55.139999   
       177660  57.990002   7222600.0    SO.csv     SO  2020    56.009998   
       177661  54.139999   9803300.0    SO.csv     SO  2020    57.990002   
       177662  50.139999   6932900.0    SO.csv     SO  2020    54.139999   

               Adj_Close_lag_1   returns  hi_lo_range  
ticker                                                 
MICT   34865               NaN       NaN     0.000000  
       34866          2.100000  0.000000     0.000000  
       34867          2.100000  0.000000     0.000000  
       34868          2.100000  0.000000     0.000000  
       34869          2.100000  0.000000     0.000000  
...                        ...       ...          ...  
SO     177658        50.139999  0.099721     5.899998  
       177659        55.139999  0.015778     5.259998  
       177660        56.009998  0.035351     2.110001  
       177661        57.990002 -0.066391     3.700001  
       177662        54.139999 -0.073883     3.170002  

[335743 rows x 14 columns]

In [96]:
pd_feat.transform(lambda x: x.assign(moving_avg=x['returns'].rolling(10).mean()))

Date       Open       High        Low      Close  \
ticker                                                                 
MICT   34865  2012-10-17   2.100000   2.100000   2.100000   2.100000   
       34866  2012-10-18   2.100000   2.100000   2.100000   2.100000   
       34867  2012-10-19   2.100000   2.100000   2.100000   2.100000   
       34868  2012-10-22   2.100000   2.100000   2.100000   2.100000   
       34869  2012-10-23   2.100000   2.100000   2.100000   2.100000   
...                  ...        ...        ...        ...        ...   
SO     177658 2020-03-26  50.000000  55.799999  49.900002  55.139999   
       177659 2020-03-27  53.220001  58.259998  53.000000  56.009998   
       177660 2020-03-30  56.720001  58.369999  56.259998  57.990002   
       177661 2020-03-31  57.000000  57.290001  53.590000  54.139999   
       177662 2020-04-01  51.900002  52.430000  49.259998  50.139999   

               Adj Close      Volume    source ticker  Year  Close_lag_1  \
ticker                                                                     
MICT   34865    2.100000         0.0  MICT.csv   MICT  2012          NaN   
       34866    2.100000         0.0  MICT.csv   MICT  2012     2.100000   
       34867    2.100000         0.0  MICT.csv   MICT  2012     2.100000   
       34868    2.100000         0.0  MICT.csv   MICT  2012     2.100000   
       34869    2.100000         0.0  MICT.csv   MICT  2012     2.100000   
...                  ...         ...       ...    ...   ...          ...   
SO     177658  55.139999   8647300.0    SO.csv     SO  2020    50.139999   
       177659  56.009998  10065000.0    SO.csv     SO  2020    55.139999   
       177660  57.990002   7222600.0    SO.csv     SO  2020    56.009998   
       177661  54.139999   9803300.0    SO.csv     SO  2020    57.990002   
       177662  50.139999   6932900.0    SO.csv     SO  2020    54.139999   

               Adj_Close_lag_1   returns  hi_lo_range  moving_avg  
ticker                                                             
MICT   34865               NaN       NaN     0.000000         NaN  
       34866          2.100000  0.000000     0.000000         NaN  
       34867          2.100000  0.000000     0.000000         NaN  
       34868          2.100000  0.000000     0.000000         NaN  
       34869          2.100000  0.000000     0.000000         NaN  
...                        ...       ...          ...         ...  
SO     177658        50.139999  0.099721     5.899998    0.010294  
       177659        55.139999  0.015778     5.259998    0.008461  
       177660        56.009998  0.035351     2.110001    0.023762  
       177661        57.990002 -0.066391     3.700001   -0.001651  
       177662        54.139999 -0.073883     3.170002   -0.007908  

[335743 rows x 15 columns]

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
 > It was not necessary but pandas should be more efficient with the RAM of a single machine and integrates well with other python libraries. 
+ Would it have been better to do it in Dask? Why?
 > in this case i don't think so, but in scenarios with much larger data sets and with out-of-core computing or to benefit from lazy evaluation it could be better.

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.